# 데이터셋별 신경망 실습
## ETTh1 · 24시간 잔차 예측

**구조:** (96,1) → Flatten → 128 → 64 → 24 + anchor  
**의도:** 직전 값 또는 직전 24시간을 기준으로 삼고 보정량을 학습합니다.

### 학습 목표
- 입력 `X`와 정답 `y`의 의미 및 shape를 확인합니다.
- 훈련·검증·테스트를 나누고 훈련 기준으로 전처리합니다.
- 모델 정의, 학습 설정, 학습 실행을 순서대로 구현합니다.
- 예측을 평가하고 모델과 전처리를 저장·복원합니다.

### 실습 방법
1. 위에서부터 순서대로 실행하세요.
2. 각 절의 **확인할 내용**과 코드 주석을 함께 읽으세요.
3. 비교 실험은 설정 셀을 수정한 뒤 데이터 준비부터 다시 실행하세요.

그래프에는 영어를 사용합니다. 기본값은 전체 학습이며, `QUICK_RUN=True`는 2 epoch 점검 모드입니다.
구성은 비교 실험의 시작점입니다.

[셀 구성·주석 방식 참고: L03 Linear Regression](https://github.com/dearyonghoon/course-ai-programming-2026/blob/83c0f113a74b737f1790f3db9f41b34885fd57b2/labs/L03_Linear_Regression.ipynb)


## 0. 라이브러리 불러오기

표 데이터 처리, 그래프, 신경망 학습에 필요한 도구를 불러옵니다.

### 확인할 내용
- 패키지가 없을 때만 아래 설치 명령의 주석을 해제하세요.


In [ ]:
# Colab에서는 보통 별도 설치가 필요 없습니다. 로컬에서는 Python 3.11/3.12 권장.
# 아래 줄의 맨 앞 # 을 지운 뒤 실행하면 현재 노트북 커널에 설치됩니다.
# %pip install "tensorflow>=2.16,<3" "scikit-learn>=1.4,<2" "pandas>=2.1,<4" "numpy>=1.26,<3" "matplotlib>=3.8,<4" joblib


In [ ]:
import os
from pathlib import Path
import json
import platform
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn import __version__ as sklearn_version
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Flatten,
    Cropping1D,
    RepeatVector,
    Add,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 작은 소수와 큰 배열의 출력을 읽기 좋게 설정
np.set_printoptions(precision=4, suppress=True)


## 1. 실험 설정

자주 바꿀 값을 한 셀에 모았습니다. 설정을 바꾸면 이후 셀을 순서대로 다시 실행합니다.

### 확인할 내용
- 학습률·배치 크기·Dropout 비율을 확인하세요.
- 빠른 점검과 전체 학습을 구분하세요.


In [ ]:
from pathlib import Path

# 같은 분할과 초기 가중치로 실험을 비교하기 위한 난수 시드
SEED = 42
# 전체 실행에서 허용할 최대 반복 횟수. 조기 종료되면 더 적게 학습합니다.
MAX_EPOCHS = 150
# 한 번의 가중치 업데이트에 사용하는 샘플 수
BATCH_SIZE = 64
# 검증 손실이 개선되지 않아도 기다릴 epoch 수
PATIENCE = 15
# optimizer가 가중치를 한 번에 얼마나 바꿀지 결정하는 학습률
LEARNING_RATE = 0.001
# 학습 중 버릴 은닉층 출력의 비율. 0이면 비활성화됩니다.
DROPOUT_RATE = 0.0
# True는 파이프라인 확인용이며, 해당 결과를 최종 성능으로 해석하지 않습니다.
# True: 2 epoch로 실행 흐름만 확인. False: 전체 학습
QUICK_RUN = False
# 학습한 모델, 전처리, 평가 지표를 데이터셋별 폴더에 저장
OUTPUT_DIR = Path("artifacts") / "etth1"
# 한 샘플에 입력할 과거 관측값의 개수
INPUT_LEN = 96
# 한 번에 예측할 미래 시점의 개수
PRED_LEN = 24
# direct / last_residual / seasonal_residual 중 예측 표현 선택
FORMULATION = "seasonal_residual"  # direct / last_residual / seasonal_residual


### 난수와 저장 폴더 준비

설정한 시드를 적용하고 실행 환경을 기록합니다.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VERSIONS = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn_version,
}
print(VERSIONS)


## 2. 데이터 살펴보기

시간순 OT 관측값을 사용합니다. 기본 설정은 과거 96시간으로 미래 24시간을 예측합니다. Drive를 연결한 뒤 FILE_PATH를 확인하세요. 로컬에서는 이 값을 실제 CSV 경로로 수정합니다.

### 확인할 내용
- 첫 5행, 데이터 크기, 열 이름을 확인하세요.
- 어떤 열을 입력과 정답으로 사용하는지 확인하세요.


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Google Colab이 아닙니다. 로컬 경로를 FILE_PATH에 지정하세요.")

# Colab의 Drive 경로입니다. 로컬에서는 실제 CSV 경로로 수정하세요.
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ETTh1.csv 위치로 수정하세요."
    )


In [ ]:
df = pd.read_csv(FILE_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
assert not df["date"].duplicated().any()
assert (
    df["date"].diff().dropna().eq(pd.Timedelta(hours=1)).all()
), "시간 간격을 먼저 확인하세요."
series = df[["OT"]].to_numpy(dtype="float32")


In [ ]:
print(df.head())
print("Rows / columns:", df.shape)


## 3. 훈련·검증·테스트 분할

시간순 분할로 미래 예측 구간을 분리합니다.

### 확인할 내용
- 훈련은 가중치 업데이트, 검증은 모델 선택, 테스트는 최종 평가에 사용합니다.


In [ ]:
n = len(series)
train_end, val_end = int(n * 0.70), int(n * 0.85)


## 4. 입력 표준화

과거 훈련 구간으로만 평균과 표준편차를 계산합니다.

### 확인할 내용
- 검증·테스트를 포함한 전체 데이터에 fit하지 않는지 확인하세요.


In [ ]:
# 시계열 scaler는 시간순 훈련 구간의 원시 값으로만 fit
scaler = StandardScaler().fit(series[:train_end])
scaled = scaler.transform(series).astype("float32")
print("Data:", df.shape, "| chronological boundaries:", train_end, val_end, n)


## 5. 시계열 window 만들기

과거 96시간으로 미래 24시간을 예측하도록 샘플을 구성합니다.

### 확인할 내용
- 입력 shape는 (샘플 수, 과거 길이, 1)입니다.
- 정답 전체가 같은 분할 안에 있어야 합니다.


In [ ]:
def make_windows(values, start, end):
    """과거 입력과 미래 정답을 만들되 정답이 지정한 시간 경계를 넘지 않게 합니다."""
    starts = np.arange(max(INPUT_LEN, start), end - PRED_LEN + 1)
    if QUICK_RUN:
        starts = starts[:: max(1, len(starts) // 128)][:128]
    if len(starts) == 0:
        raise ValueError("window를 만들 데이터가 부족합니다.")
    X = np.stack([values[t - INPUT_LEN : t] for t in starts])
    y = np.stack([values[t : t + PRED_LEN, 0] for t in starts])
    assert starts.min() >= start and np.max(starts + PRED_LEN) <= end
    return X.astype("float32"), y.astype("float32"), starts


X_train, y_train, train_starts = make_windows(scaled, 0, train_end)
X_val, y_val, val_starts = make_windows(scaled, train_end, val_end)
X_test, y_test, test_starts = make_windows(scaled, val_end, n)
assert X_train.shape[1:] == (INPUT_LEN, 1)
assert y_train.shape[1] == PRED_LEN
assert val_starts[0] == train_end and test_starts[0] == val_end
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)


> ### 체크포인트
> 검증·테스트를 이용해 전처리 통계를 다시 계산하지 않았는지 확인하세요.


## 6. 신경망 모델 정의

Input → 은닉층 → 출력층 순서로 네트워크를 구성합니다. 시계열 잔차 모델은 기준 예측에 보정량을 더합니다.

### 확인할 내용
- summary에서 각 층의 출력 shape와 파라미터 수를 확인하세요.


In [ ]:
if FORMULATION not in {"direct", "last_residual", "seasonal_residual"}:
    raise ValueError("FORMULATION 값을 확인하세요.")
if not 0 < PRED_LEN <= INPUT_LEN:
    raise ValueError("0 < PRED_LEN <= INPUT_LEN 이어야 합니다.")
# 배치 축은 생략하고 샘플 하나의 입력 shape를 지정
inputs = Input(shape=(INPUT_LEN, 1), name="history_ot")
h = Flatten()(inputs)
# 순서대로 두 은닉층을 연결합니다.
h = Dense(128, activation="relu", kernel_initializer="he_normal")(h)
if DROPOUT_RATE > 0:
    h = Dropout(DROPOUT_RATE)(h)
h = Dense(64, activation="relu", kernel_initializer="he_normal")(h)
if DROPOUT_RATE > 0:
    h = Dropout(DROPOUT_RATE)(h)
# 잔차 모델의 마지막 층은 0 초기화하여 시작 예측이 기준선과 같도록 설정
correction = Dense(
    PRED_LEN,
    kernel_initializer="glorot_uniform" if FORMULATION == "direct" else "zeros",
    bias_initializer="zeros",
    name="forecast_or_correction",
)(h)
if FORMULATION == "seasonal_residual":
    anchor = Flatten()(Cropping1D((INPUT_LEN - PRED_LEN, 0))(inputs))
    outputs = Add(name="anchor_plus_correction")([anchor, correction])
elif FORMULATION == "last_residual":
    last = Flatten()(Cropping1D((INPUT_LEN - 1, 0))(inputs))
    anchor = Flatten()(RepeatVector(PRED_LEN)(last))
    outputs = Add(name="anchor_plus_correction")([anchor, correction])
else:
    outputs = correction
model = Model(inputs, outputs)

# 각 층의 출력 shape와 학습할 파라미터 수 확인
model.summary()


## 7. 손실 함수와 optimizer 설정

회귀 출력에는 확률 활성화를 쓰지 않습니다. 표준화한 정답에 대한 MSE를 최소화합니다. compile은 설정만 하며, 실제 가중치 갱신은 fit에서 시작됩니다.

### 확인할 내용
- 변환된 정답 단위의 MSE로 학습한다는 점을 확인하세요.


In [ ]:
# optimizer, loss, 관찰할 metric을 설정합니다. 아직 학습하지 않습니다.
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])


# 잔차 head가 0일 때 시작 예측은 anchor와 같습니다.
if FORMULATION != "direct":
    expected = (
        X_val[:2, -PRED_LEN:, 0]
        if FORMULATION == "seasonal_residual"
        else np.repeat(X_val[:2, -1, 0:1], PRED_LEN, axis=1)
    )
    np.testing.assert_allclose(model.predict(X_val[:2], verbose=0), expected, atol=1e-6)


## 8. 모델 학습

검증 손실을 관찰하며 학습하고, 가장 좋았던 가중치를 복원합니다.

### 확인할 내용
- EarlyStopping의 patience와 restore_best_weights를 확인하세요.


In [ ]:
# 검증 손실이 정체되면 학습률을 줄이고, 더 개선되지 않으면 학습 종료
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
]

# MSE(anchor + correction, y)는 MSE(correction, y - anchor)와 같습니다.


### 학습 실행

batch_size는 업데이트당 샘플 수, epochs는 전체 데이터를 반복할 최대 횟수입니다.


In [ ]:
# fit은 반복 학습을 수행하고 epoch별 손실을 History에 기록
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=2 if QUICK_RUN else MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=2,
)


## 9. 학습 곡선 확인

훈련 손실과 검증 손실을 같은 그래프에서 비교합니다.

### 확인할 내용
- 훈련 손실만 감소하고 검증 손실이 커지는 구간이 있는지 확인하세요.


In [ ]:
# loss는 훈련 손실, val_loss는 가중치 업데이트에 쓰지 않은 검증 손실입니다.
epochs_run = np.arange(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(7, 3))
plt.plot(epochs_run, history.history["loss"], label="Train")
plt.plot(epochs_run, history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=150)
plt.show()


> ### 체크포인트
> History에 실제로 학습한 epoch 수와 훈련·검증 손실이 기록되어 있는지 확인하세요.


## 10. 테스트 데이터로 평가

분류는 예측 라벨을 평가하고, 회귀는 예측을 원래 단위로 되돌린 뒤 평가합니다.

### 확인할 내용
- 기준선과 모델의 지표를 비교하세요.
- 테스트 결과로 설정을 다시 고르지 마세요.


In [ ]:
def inverse_ot(values):
    """원래 배열 shape를 유지하면서 OT 표준화를 역변환합니다."""
    shape = values.shape
    return scaler.inverse_transform(values.reshape(-1, 1)).reshape(shape)


# 신경망 출력은 변환된 단위이므로 역변환한 뒤 실제 정답과 비교
pred_s = model.predict(X_test, verbose=0)  # anchor가 이미 포함된 최종 예측
# 표준화한 정답을 원래 OT 단위로 복원
actual = inverse_ot(y_test)
# 분류에서는 예측 라벨, 회귀에서는 원래 단위의 예측값
pred = inverse_ot(pred_s)
# 모든 미래 시점에 마지막 관측값을 반복하는 기준선
last_pred = inverse_ot(np.repeat(X_test[:, -1, 0:1], PRED_LEN, axis=1))
# 직전 24시간의 관측 패턴을 반복하는 기준선
seasonal_pred = inverse_ot(X_test[:, -PRED_LEN:, 0])
records = []
for name, values in [
    ("MLP", pred),
    ("Last value", last_pred),
    ("Seasonal naive 24h", seasonal_pred),
]:
    records.append(
        {
            "model": name,
            "mae_celsius": float(mean_absolute_error(actual, values)),
            "mse_celsius_squared": float(mean_squared_error(actual, values)),
        }
    )
metrics = {"formulation": FORMULATION, "test_results": records}
print(pd.DataFrame(records).to_string(index=False))
# 멀리 예측할수록 오차가 어떻게 달라지는지 시점별로 계산
horizon_mae = np.mean(np.abs(actual - pred), axis=0)


### 평가 결과 시각화

정답과 예측의 관계를 그래프로 확인합니다. 분류에서는 혼동행렬을 읽어 보세요.


In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(np.arange(1, PRED_LEN + 1), horizon_mae)
plt.xlabel("Horizon (hour)")
plt.ylabel("MAE (C)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "horizon_mae.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(np.arange(-INPUT_LEN, 0), inverse_ot(X_test[0, :, 0]), label="History")
plt.plot(np.arange(PRED_LEN), actual[0], label="Actual")
plt.plot(np.arange(PRED_LEN), pred[0], label="MLP")
plt.legend()
plt.xlabel("Time step")
plt.ylabel("OT (C)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "forecast.png", dpi=150)
plt.show()


## 11. 모델과 전처리 저장

신경망뿐 아니라 전처리 통계와 입력 열의 순서도 함께 저장합니다.

### 확인할 내용
- .keras 파일과 preprocessing.joblib 파일이 만들어졌는지 확인하세요.


In [ ]:
def write_json(filename, value):
    """평가 결과와 설정을 UTF-8 JSON으로 저장합니다."""
    (OUTPUT_DIR / filename).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )


In [ ]:
# 학습한 가중치와 모델 구조를 하나의 .keras 파일에 저장
model.save(OUTPUT_DIR / "model.keras")
# 모델과 함께 전처리 통계·입출력 열 순서도 저장
joblib.dump(
    {
        "scaler": scaler,
        "input_len": INPUT_LEN,
        "pred_len": PRED_LEN,
        "feature_columns": ["OT"],
        "formulation": FORMULATION,
    },
    OUTPUT_DIR / "preprocessing.joblib",
)


### 평가 기록 저장

손실 기록, 설정, 테스트 예측값을 파일로 남깁니다.


In [ ]:
write_json("metrics.json", metrics)
write_json(
    "config.json",
    {
        "seed": SEED,
        "formulation": FORMULATION,
        "input_len": INPUT_LEN,
        "pred_len": PRED_LEN,
        "train_end": train_end,
        "val_end": val_end,
        "quick_run": QUICK_RUN,
        "versions": VERSIONS,
    },
)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "history.csv", index=False)
pd.DataFrame(
    {
        "target_start": np.repeat(test_starts, PRED_LEN),
        "horizon_hour": np.tile(np.arange(1, PRED_LEN + 1), len(test_starts)),
        "actual_ot": actual.ravel(),
        "predicted_ot": pred.ravel(),
    }
).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## 12. 저장한 모델로 새 샘플 예측

같은 전처리를 적용한 뒤 모델을 호출하고, 필요한 역변환을 수행합니다.

### 확인할 내용
- 저장 전후의 예측이 같아야 합니다.
- 원시 입력을 scaler 없이 모델에 직접 넣지 마세요.


In [ ]:
# 저장한 모델과 전처리를 함께 복원하여 동일한 추론 흐름 사용
loaded = load_model(OUTPUT_DIR / "model.keras")
bundle = joblib.load(OUTPUT_DIR / "preprocessing.joblib")
# 마지막 관측 구간으로 다음 시간대의 값을 예측
recent_ot = series[-bundle["input_len"] :]
recent_scaled = bundle["scaler"].transform(recent_ot).astype("float32")[None, ...]
future_s = loaded.predict(recent_scaled, verbose=0)
# 저장 전후의 예측이 수치 오차 범위에서 같은지 확인
np.testing.assert_allclose(
    future_s, model.predict(recent_scaled, verbose=0), rtol=1e-5, atol=1e-6
)
future = bundle["scaler"].inverse_transform(future_s.reshape(-1, 1)).ravel()
print("Forecast for next hours:", future)
print("Saved to:", OUTPUT_DIR.resolve())


> ### 체크포인트
> 전처리 → 모델 예측 → 역변환 순서를 처음 학습할 때와 동일하게 유지했는지 확인하세요.


## 직접 해보기

**다음 비교:** FORMULATION만 바꾸어 validation loss를 비교한 뒤 test를 확인하세요.
평가는 각 예측 시점까지 실제 관측값을 이용하는 rolling-origin 방식입니다.
window끼리 target 시점이 겹치므로 모든 예측 원점·horizon의 평균 지표입니다.


## 정리

**데이터 확인 → 분할 → 훈련 기준 전처리 → 모델 정의 → compile → fit → 평가 → 저장·복원**

- `X_train`과 `y_train`으로 가중치를 학습합니다.
- 검증 데이터로 학습을 멈출 시점과 설정을 고릅니다.
- 테스트는 최종 평가에 사용합니다.
